In [ ]:
!pip install diffusers transformers accelerate safetensors torch torchvision gradio --quiet

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
import gradio as gr

In [ ]:
HF_TOKEN = "My_HF_Token" # Add your Token here in order to get output

In [ ]:
print("Loading model...")

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    use_auth_token=HF_TOKEN
)

pipe = pipe.to(device)

print("Model loaded successfully!")

In [ ]:
def generate_images(prompt, negative_prompt, steps, guidance, num_images):

    images = []

    for i in range(num_images):

        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=steps,
            guidance_scale=guidance
        )

        images.append(result.images[0])

    return images

In [ ]:
def correct_image(previous_prompt, correction_text, steps, guidance): # this feature is under development

    # Combine old prompt with correction
    new_prompt = previous_prompt + ", " + correction_text

    result = pipe(
        prompt=new_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance
    )

    return result.images[0]

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("# 🎨 AI Image Generator")

    with gr.Tab("Generate Images"):

        prompt = gr.Textbox(label="Prompt")

        negative_prompt = gr.Textbox(
            label="Negative Prompt",
            value="blurry, low quality, distorted face"
        )

        steps = gr.Slider(20, 60, value=30, label="Inference Steps")

        guidance = gr.Slider(5, 12, value=7.5, label="Guidance Scale")

        num_images = gr.Slider(1, 5, value=2, step=1, label="Number of Images")

        generate_btn = gr.Button("Generate")

        gallery = gr.Gallery(label="Generated Images")

        generate_btn.click(
            generate_images,
            inputs=[prompt, negative_prompt, steps, guidance, num_images],
            outputs=gallery
        )


    with gr.Tab("Correct Previous Image"):

        prev_prompt = gr.Textbox(label="Previous Prompt")

        correction = gr.Textbox(
            label="Correction (Example: more realistic face, better lighting)"
        )

        c_steps = gr.Slider(20, 60, value=35, label="Steps")

        c_guidance = gr.Slider(5, 12, value=8, label="Guidance")

        correct_btn = gr.Button("Correct Image")

        corrected_output = gr.Image(label="Corrected Image")

        correct_btn.click(
            correct_image,
            inputs=[prev_prompt, correction, c_steps, c_guidance],
            outputs=corrected_output
        )

In [ ]:
demo.launch()